# k4 — Chỉ mục dense, 9 nhánh (vi/en/zh × Dense/LSR/BM25), RRF, chẩn đoán không cần nhãn

**Cài đặt:** Accelerator = **GPU T4** (chỉ để mã hoá câu hỏi), Internet = On.
**Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks`, `mir2-qtrans`, mọi `mir2-emb-m*`.
`USE` = bản dịch dùng cho nhánh en/zh (tên thư mục trong qtrans/, chọn sau khi chấm bản dịch).
RAM: IVF-SQ8 ~1 KB/vector (đoạn con ~6,5 triệu → ~6,5 GB) — vừa 30 GB. `SAVE_INDEX = False` để không lưu file chỉ mục
(~9 GB) vào output nếu không cần.
**Đầu ra:** `runs/` (9 nhánh, các mức @parents/@children, rrf9 …), `qtrans/queries_*.parquet`, `qemb/`,
`eval/diag/diag.md` → dataset **`mir2-runs`**.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k4-v1 (15/10/2026)'
USE = 'qwen25_14b'
SAVE_INDEX = False
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
def _ver(h):                                                  # output notebook cũ (vd mir2-chunks/code) cũng có mirret/
    v = os.path.join(os.path.dirname(os.path.dirname(h)), 'VERSION.txt')
    return open(v, encoding='utf-8').read().strip() if os.path.exists(v) else ''
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True),
              key=lambda h: (_ver(h), -len(h)), reverse=True)   # bản mã mới nhất theo VERSION.txt (ngày giờ)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks',))

In [ ]:
# (2) Câu hỏi 3 ngôn ngữ: chép bản dịch từ mir2-qtrans rồi dựng qtrans/queries_{vi,en,zh}.parquet
for d in locate_all('qtrans'):
    for f in glob.glob(f'{d}/qtrans/*/records*.jsonl'):
        dst = f'{K.work}/qtrans/' + os.path.relpath(f, f'{d}/qtrans')
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst)
sh(f'python -m mirret.translate build --config {K.cfg} --use {USE}')
bm = locate('index/bm25', required=False)
if bm is not None:
    K.set({'bm25.index_dir': str(bm / 'index' / 'bm25')})
EMB = ','.join(sorted({str(d / 'emb') for d in locate_all('emb')}))
print('emb:', EMB)

In [ ]:
# (3) Chỉ mục dense từng mức + mã hoá câu hỏi (GPU)
for lv in ('parents', 'children'):
    sh(f'python -m mirret.index dense --config {K.cfg} --level {lv} --emb {EMB}', f'dense_{lv}')
sh(f'python -m mirret.retrieve qencode --config {K.cfg} --device cuda:0', 'qencode')

In [ ]:
# (4) 9 nhánh + gộp RRF + chẩn đoán không cần nhãn (trên cả 1.200 câu)
sh(f'python -m mirret.retrieve run --config {K.cfg} --emb {EMB}', 'retrieve')
sh(f'python -m mirret.fuse --config {K.cfg}', 'fuse')
sh(f'python -m mirret.diag --config {K.cfg}', 'diag')
if not SAVE_INDEX:
    for f in glob.glob(f'{K.work}/index/dense_*'):
        os.remove(f)
report('k4')